# PairRecon column mapping

This notebook discovers and reviews the column mappings for one Pair. It submits the complete approved list in one SQL statement. With `overwrite=True`, the entire column mapping table is replaced by this list, including removal of mappings for other Pairs. Set `overwrite=False` to insert or update only the supplied attributes.


In [ ]:
from pathlib import Path

import pandas as pd

from pair_recon.app.config import env_value, load_environment
from pair_recon.app.runner import StatementSql, pair_config, workspace
from pair_recon.core.metadata.constant import CATALOG_SCHEMA_ENV, catalog_tables
from pair_recon.core.metadata.mapping import (
    automatch_columns,
    discover_columns,
    discover_databricks_partition,
    discover_relations,
    get_date_column_type,
    upsert_column_mappings,
)
from pair_recon.core.metadata.workbook import read_pair_workbook

load_environment()
PROFILE = env_value("DBX_PROFILE", default="corp")
WAREHOUSE_ID = env_value("DBX_WAREHOUSE_ID", default="")
CATALOG_SCHEMA = env_value(CATALOG_SCHEMA_ENV, default="catalog.schema")
TABLES = catalog_tables(CATALOG_SCHEMA)
PAIR_TABLE = TABLES.pair_meta
PAIR_NAME = ""
WORKBOOK = Path(env_value("DBX_PAIR_WORKBOOK", default="setup/pairs.xlsx")).expanduser()

In [ ]:
rows = {row["pair_name"]: row for row in read_pair_workbook(WORKBOOK)}
if PAIR_NAME not in rows:
    raise ValueError(f"PAIR_NAME must be one of: {', '.join(sorted(rows))}")
row = rows[PAIR_NAME]
effective = pair_config(row)
client = workspace(PROFILE)
sql = StatementSql(client, WAREHOUSE_ID)
COLUMN_MAP_TABLE = TABLES.column_map

In [ ]:
print(discover_relations(effective["left"], sql=sql))
print(discover_relations(effective["right"], sql=sql))
if effective["right"].get("source") == "databricks":
    print({"partition_column": discover_databricks_partition(sql, effective["right"])})

left_columns = discover_columns(effective["left"], sql=sql)
right_columns = discover_columns(effective["right"], sql=sql)
candidates = pd.DataFrame(automatch_columns(left_columns, right_columns))
print(get_date_column_type(effective["left"], left_columns))
print(get_date_column_type(effective["right"], right_columns, sql))
display(candidates.iloc[:20])

Review unmatched or ambiguous rows above. Add, remove, or edit rows in `approved` and set an optional `data_type` before applying. With `OVERWRITE = True`, `approved` replaces the entire column mapping table. An empty list clears the whole table. Use `OVERWRITE = False` to preserve all omitted mappings.


In [ ]:
approved = candidates.loc[
    candidates["status"] == "matched",
    ["left_column", "right_column", "data_type"],
].copy()
# Example manual correction:
# approved.loc[approved["left_column"] == "AMOUNT", "data_type"] = "numeric"
display(approved)

In [ ]:
APPLY = False
OVERWRITE = True  # True replaces the entire mapping table; False applies a partial update.
if APPLY:
    result = upsert_column_mappings(
        sql, COLUMN_MAP_TABLE, PAIR_NAME, approved.to_dict(orient="records"),
        overwrite=OVERWRITE
    )
    display(result)
else:
    print("Preview only. Set APPLY = True after reviewing approved mappings.")